In [1]:
import requests
import pandas as pd
import numpy as np
import holidays

# Define date ranges
train_start_date = '2015-01-01'
train_end_date = '2023-12-31'

# List of bidding zones
bidding_zones = [
    "AT", "BE", "CH", "CZ", "DK1", "DK2", "FR", "HU", "IT-North", 
    "NL", "NO2", "PL", "SE4", "SI"
]
# AT (Austria)
# BE (Belgium)
# CH (Switzerland)
# CZ (Czech Republic)
# DK1 (Denmark 1)
# DK2 (Denmark 2)
# FR (France)
# HU (Hungary)
# IT-North (Italy North)
# NL (Netherlands)
# NO2 (Norway 2)
# PL (Poland)
# SE4 (Sweden 4)
# SI (Slovenia)
### Not included as data already fetched/crawled and exists
# DE-LU (Germany, Luxembourg)
### Not included as error when running
# DE-AT-LU (Germany, Austria, Luxembourg)

# Define the cyclic encoding function
def cyclicEncode(data, col, max_val):
    data[col + '_sin'] = np.sin(2 * np.pi * data[col] / max_val)
    data[col + '_cos'] = np.cos(2 * np.pi * data[col] / max_val)
    return data

# Add calendar features
def add_calendar_features(df):
    german_holidays = holidays.Germany(years=df['ds'].dt.year.unique())
    
    # Basic date features
    df['day_of_week'] = df['ds'].dt.dayofweek  # 0=Monday, 6=Sunday
    df['month'] = df['ds'].dt.month
    df['hour'] = df['ds'].dt.hour
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    
    # Holiday feature
    df['is_holiday'] = df['ds'].apply(lambda x: int(x in german_holidays))
    
    # Add cyclic encoding for month, day_of_week, and hour
    df = cyclicEncode(df, 'month', 12)
    df = cyclicEncode(df, 'day_of_week', 7)
    df = cyclicEncode(df, 'hour', 24)
    
    return df

# Loop over all bidding zones
for bidding_zone in bidding_zones:
    print(f"Processing data for: {bidding_zone}")
    
    # Fetch data from the API
    url_train = f"https://api.energy-charts.info/price?bzn={bidding_zone}&start={train_start_date}&end={train_end_date}"
    response_train = requests.get(url_train)
    
    if response_train.status_code != 200:
        print(f"Failed to fetch data for {bidding_zone}, skipping...")
        continue
    
    data_train = response_train.json()
    
    # Convert timestamps to UTC, then to Germany time (Europe/Berlin)
    train_timestamps = pd.to_datetime(data_train['unix_seconds'], unit='s', utc=True).tz_convert('Europe/Berlin')
    
    # Convert to DataFrame
    train_df = pd.DataFrame({
        'timestamp': train_timestamps,
        'price': data_train['price']
    })
    
    # Remove timezone information while keeping local time
    train_df['timestamp'] = train_df['timestamp'].dt.tz_localize(None)
    
    # Clean the data (remove rows with null prices)
    train_df = train_df.dropna()
    
    # Rename columns for NeuralForecast
    train_df = train_df.rename(columns={'timestamp': 'ds', 'price': 'y'})
    
    # Add unique_id column
    train_df['unique_id'] = bidding_zone
    
    # Add calendar features
    train_df = add_calendar_features(train_df)
    
    # Save final processed DataFrame to CSV
    train_df.to_csv(f'{bidding_zone}_training_data_with_calendar.csv', index=False)
    
    # Check date ranges
    print(f"First date in train_df for {bidding_zone}:", train_df['ds'].min())
    print(f"Last date in train_df for {bidding_zone}:", train_df['ds'].max())
    print("-------------------------------------------")
    
print("Data processing complete for all bidding zones.")


Processing data for: AT
First date in train_df for AT: 2018-10-01 00:00:00
Last date in train_df for AT: 2023-12-31 23:00:00
-------------------------------------------
Processing data for: BE
First date in train_df for BE: 2015-01-05 00:00:00
Last date in train_df for BE: 2023-12-31 23:00:00
-------------------------------------------
Processing data for: CH
First date in train_df for CH: 2015-01-01 01:00:00
Last date in train_df for CH: 2023-12-31 23:00:00
-------------------------------------------
Processing data for: CZ
First date in train_df for CZ: 2015-01-01 01:00:00
Last date in train_df for CZ: 2023-12-31 23:00:00
-------------------------------------------
Processing data for: DK1
First date in train_df for DK1: 2015-01-01 01:00:00
Last date in train_df for DK1: 2023-12-31 23:00:00
-------------------------------------------
Processing data for: DK2
First date in train_df for DK2: 2015-01-01 01:00:00
Last date in train_df for DK2: 2023-12-31 23:00:00
------------------------